In [4]:
# ============================================================================
# STEPS 1 & 2: CREATE MULTI-CELEBRITY DATASET 
# ============================================================================
import os, numpy as np, pandas as pd, kagglehub
from PIL import Image, ImageEnhance
import random

# ============================================================================
# CONFIGURATION
# ============================================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)




# ============================================================================
# ENSURE DIRECTORIES EXIST
# ============================================================================
import os

YOLO_DATASET_DIR = "./yolo_celeba_dataset"

# Create all necessary directories
os.makedirs(os.path.join(YOLO_DATASET_DIR, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(YOLO_DATASET_DIR, "labels", "val"), exist_ok=True)

print(f"Created directories in: {os.path.abspath(YOLO_DATASET_DIR)}")

# ============================================================================
# GENERATE TRAINING DATASET
# ============================================================================
NUM_TRAIN = 1000
NUM_VAL = 200

print(f"\nGenerating {NUM_TRAIN} training composite images...")

for i in range(NUM_TRAIN):
    composite, boxes = create_composite_image(df_target)
    
    if len(boxes) == 0:
        continue
    
    composite = augment_composite(composite)
    
    # Save image
    img_filename = f"train_{i:05d}.jpg"
    img_path = os.path.join(YOLO_DATASET_DIR, "images", "train", img_filename)
    composite.save(img_path)
    
    
# class group celebrity IDs
target_celebs = [
    4126, 7904, 8656, 9319, 3321, 8968, 2820, 3227, 9063, 8871,
    7282, 8945, 3782, 8722, 3401, 1964, 4561, 2880, 3745, 3699,
    9152, 9256, 2463, 2562, 3431, 1499, 8045, 10173, 2522, 228,
    5239, 2425, 4304, 5260, 2837, 1158, 3698, 6098, 6568, 9151,
    800, 619, 487, 1852, 8265, 447, 10046
]

print("="*70)
print("Creating Multi-Celebrity Dataset for YOLOv8")
print("="*70)
print(f"Target celebrities: {len(target_celebs)}")

# ============================================================================
# LOAD CELEBA DATA
# ============================================================================
print("\nLoading CelebA dataset...")
root = kagglehub.dataset_download("jessicali9530/celeba-dataset")
img_dirs = [
    os.path.join(root, "img_align_celeba", "img_align_celeba"),
    os.path.join(root, "img_align_celeba")
]
IMG_DIR = next((p for p in img_dirs if os.path.isdir(p)), None)

# Load identity file from Downloads
ID_PATH = "identity_CelebA.txt"
if not os.path.exists(ID_PATH):
    print(f"ERROR: {ID_PATH} not found!")
    print("Please place identity_CelebA.txt in /Users/Betty/Downloads/")
    raise FileNotFoundError("identity_CelebA.txt missing")

df = pd.read_csv(ID_PATH, sep=r'\s+', header=None, names=['filename', 'identity'])
print(f"Total images in CelebA: {len(df):,}")

# Filter to target celebrities
df_target = df[df['identity'].isin(target_celebs)].copy()
found_celebs = sorted(df_target['identity'].unique())
missing = set(target_celebs) - set(found_celebs)

if missing:
    print(f"\nWarning: {len(missing)} IDs not found: {sorted(missing)}")
    target_celebs = found_celebs

print(f"Using {len(target_celebs)} celebrities with {len(df_target):,} total images")

# Create class mapping (0 to N-1)
NUM_CELEBS = len(target_celebs)
class_map = {celeb_id: idx for idx, celeb_id in enumerate(sorted(target_celebs))}
reverse_class_map = {idx: celeb_id for celeb_id, idx in class_map.items()}

# ============================================================================
# FUNCTION: CREATE COMPOSITE IMAGE
# ============================================================================
def create_composite_image(df_source, canvas_size=(640, 640), min_celebs=2, max_celebs=5):
    """Create composite with multiple celebrity faces"""
    canvas = Image.new('RGB', canvas_size, color=(128, 128, 128))
    
    num_celebs = random.randint(min_celebs, max_celebs)
    sampled_celebs = random.sample(target_celebs, min(num_celebs, len(target_celebs)))
    
    boxes = []
    occupied_regions = []
    
    for celeb_id in sampled_celebs:
        celeb_imgs = df_source[df_source['identity'] == celeb_id]['filename'].tolist()
        if not celeb_imgs:
            continue
        
        img_path = os.path.join(IMG_DIR, random.choice(celeb_imgs))
        
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            continue
        
        # Random scale
        scale = random.uniform(0.3, 0.7)
        new_w = int(img.width * scale)
        new_h = int(img.height * scale)
        img_resized = img.resize((new_w, new_h))
        
        # Find non-overlapping position
        placed = False
        for attempt in range(10):
            x = random.randint(0, max(0, canvas_size[0] - new_w))
            y = random.randint(0, max(0, canvas_size[1] - new_h))
            
            overlap = False
            for (ox, oy, ow, oh) in occupied_regions:
                if not (x + new_w < ox or x > ox + ow or 
                       y + new_h < oy or y > oy + oh):
                    overlap = True
                    break
            
            if not overlap or attempt == 9:
                placed = True
                break
        
        if placed:
            canvas.paste(img_resized, (x, y))
            occupied_regions.append((x, y, new_w, new_h))
            
            # YOLO format: normalized coordinates
            x_center = (x + new_w / 2) / canvas_size[0]
            y_center = (y + new_h / 2) / canvas_size[1]
            width = new_w / canvas_size[0]
            height = new_h / canvas_size[1]
            
            class_id = class_map[celeb_id]
            boxes.append([class_id, x_center, y_center, width, height])
    
    return canvas, boxes

# ============================================================================
# FUNCTION TO AUGMENT COMPOSITE
# ============================================================================
def augment_composite(img):
    """Apply random augmentations"""
    if random.random() > 0.5:
        enhancer = ImageEnhance.Brightness(img)
        img = enhancer.enhance(random.uniform(0.8, 1.2))
    
    if random.random() > 0.5:
        enhancer = ImageEnhance.Contrast(img)
        img = enhancer.enhance(random.uniform(0.9, 1.1))
    
    if random.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    
    return img

# ============================================================================
# GENERATE TRAINING DATASET
# ============================================================================
NUM_TRAIN = 1000
NUM_VAL = 200

print(f"\nGenerating {NUM_TRAIN} training composite images...")

for i in range(NUM_TRAIN):
    composite, boxes = create_composite_image(df_target)
    
    if len(boxes) == 0:
        continue
    
    composite = augment_composite(composite)
    
    # Save image
    img_filename = f"train_{i:05d}.jpg"
    img_path = os.path.join(YOLO_DATASET_DIR, "images", "train", img_filename)
    composite.save(img_path)
    
    # Save labels
    label_filename = f"train_{i:05d}.txt"
    label_path = os.path.join(YOLO_DATASET_DIR, "labels", "train", label_filename)
    
    with open(label_path, 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    if (i + 1) % 100 == 0:
        print(f"  Generated {i+1}/{NUM_TRAIN}")

# ============================================================================
# GENERATE VALIDATION DATASET
# ============================================================================
print(f"\nGenerating {NUM_VAL} validation composite images...")

for i in range(NUM_VAL):
    composite, boxes = create_composite_image(df_target)
    
    if len(boxes) == 0:
        continue
    
    img_filename = f"val_{i:05d}.jpg"
    img_path = os.path.join(YOLO_DATASET_DIR, "images", "val", img_filename)
    composite.save(img_path)
    
    label_filename = f"val_{i:05d}.txt"
    label_path = os.path.join(YOLO_DATASET_DIR, "labels", "val", label_filename)
    
    with open(label_path, 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    if (i + 1) % 50 == 0:
        print(f"  Generated {i+1}/{NUM_VAL}")

# ============================================================================
# CREATE data.yaml FOR YOLOV8
# ============================================================================
yaml_content = f"""path: {YOLO_DATASET_DIR}
train: images/train
val: images/val

nc: {NUM_CELEBS}
names: {list(range(NUM_CELEBS))}
"""

yaml_path = os.path.join(YOLO_DATASET_DIR, "data.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

# Save celebrity mapping
mapping_df = pd.DataFrame({
    'class_id': list(reverse_class_map.keys()),
    'celebrity_id': list(reverse_class_map.values())
})
mapping_df.to_csv(os.path.join(YOLO_DATASET_DIR, "class_mapping.csv"), index=False)

print("\n" + "="*70)
print("DATASET CREATION COMPLETE!")
print("="*70)
print(f"Dataset: {YOLO_DATASET_DIR}")
print(f"Training images: {NUM_TRAIN}")
print(f"Validation images: {NUM_VAL}")
print(f"Classes: {NUM_CELEBS} celebrities")
print("\nReady for YOLOv8 training!")

Created directories in: /courses/IE7615.202610/students/tegegne.r/yolo_celeba_dataset

Generating 1000 training composite images...
Creating Multi-Celebrity Dataset for YOLOv8
Target celebrities: 47

Loading CelebA dataset...
Total images in CelebA: 202,599
Using 47 celebrities with 1,328 total images

Generating 1000 training composite images...
  Generated 100/1000
  Generated 200/1000
  Generated 300/1000
  Generated 400/1000
  Generated 500/1000
  Generated 600/1000
  Generated 700/1000
  Generated 800/1000
  Generated 900/1000
  Generated 1000/1000

Generating 200 validation composite images...
  Generated 50/200
  Generated 100/200
  Generated 150/200
  Generated 200/200

DATASET CREATION COMPLETE!
Dataset: ./yolo_celeba_dataset
Training images: 1000
Validation images: 200
Classes: 47 celebrities

Ready for YOLOv8 training!


In [8]:
# ============================================================================
# STEP 3 & 4: TRAIN YOLOV8 AND RUN TEST
# ============================================================================
import os
import pandas as pd
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# ============================================================================
# LOAD DATASET CONFIGURATION
# ============================================================================
YOLO_DATASET_DIR = "./yolo_celeba_dataset"

print("="*70)
print("Loading existing dataset...")
print("="*70)

# Load class mapping
class_mapping = pd.read_csv(os.path.join(YOLO_DATASET_DIR, "class_mapping.csv"))
NUM_CELEBS = len(class_mapping)
reverse_class_map = dict(zip(class_mapping['class_id'], class_mapping['celebrity_id']))

# Count images
train_img_dir = os.path.join(YOLO_DATASET_DIR, "images", "train")
val_img_dir = os.path.join(YOLO_DATASET_DIR, "images", "val")

NUM_TRAIN = len([f for f in os.listdir(train_img_dir) if f.endswith('.jpg')])
NUM_VAL = len([f for f in os.listdir(val_img_dir) if f.endswith('.jpg')])

print(f"Dataset: {os.path.abspath(YOLO_DATASET_DIR)}")
print(f"Classes: {NUM_CELEBS} celebrities")
print(f"Training images: {NUM_TRAIN}")
print(f"Validation images: {NUM_VAL}")

# ============================================================================
# STEP 3: TRAIN YOLOV8
# ============================================================================
print("\n" + "="*70)
print("Training YOLOv8 for Celebrity Detection")
print("="*70)

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

print("\nStarting training on CPU...")
print("Note: This will take 1-2 hours on CPU")

# Train the model
results = model.train(
    data=os.path.join(YOLO_DATASET_DIR, 'data.yaml'),
    epochs=20,
    imgsz=416,
    batch=4,
    name='celeb_detector',
    patience=10,
    save=True,
    device='cpu',
    project='runs/detect',
    plots=True,
    verbose=True
)

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print(f"Best model saved: runs/detect/celeb_detector/weights/best.pt")

# ============================================================================
# STEP 4: INFERENCE FUNCTIONS
# ============================================================================
print("\n" + "="*70)
print("Setting up inference functions...")
print("="*70)

# Load trained model
trained_model = YOLO('runs/detect/celeb_detector/weights/best.pt')

def detect_celebrities(image_path, conf_threshold=0.25):
    """
    Detect and identify celebrities in an image
    
    Args:
        image_path: Path to image
        conf_threshold: Confidence threshold (0-1)
    
    Returns:
        List of detections with celebrity IDs, confidence, and bounding boxes
    """
    results = trained_model(image_path, conf=conf_threshold)
    
    detections = []
    for result in results:
        boxes = result.boxes
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            confidence = box.conf[0].item()
            class_id = int(box.cls[0].item())
            celebrity_id = reverse_class_map[class_id]
            
            detections.append({
                'celebrity_id': celebrity_id,
                'confidence': confidence,
                'bbox': [int(x1), int(y1), int(x2), int(y2)]
            })
    
    return detections

def visualize_detections(image_path, detections, save_path=None):
    """
    Draw bounding boxes and labels on image
    
    Args:
        image_path: Path to image
        detections: List of detections from detect_celebrities()
        save_path: Optional path to save visualization
    
    Returns:
        PIL Image with visualizations
    """
    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    
    try:
        font = ImageFont.truetype("arial.ttf", 16)
    except:
        font = ImageFont.load_default()
    
    colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'pink', 'cyan']
    
    for i, det in enumerate(detections):
        x1, y1, x2, y2 = det['bbox']
        color = colors[i % len(colors)]
        
        # Draw bounding box
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        
        # Draw label
        label = f"ID {det['celebrity_id']}: {det['confidence']:.2f}"
        draw.text((x1, y1 - 20), label, fill=color, font=font)
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Detected {len(detections)} celebrities")
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    
    return img

def process_image(image_path):
    """
    Complete pipeline: detect and visualize celebrities in any image
    
    Args:
        image_path: Path to image
    
    Returns:
        List of detections
    """
    print(f"\nProcessing: {image_path}")
    
    if not os.path.exists(image_path):
        print("Error: Image not found")
        return []
    
    # Detect
    detections = detect_celebrities(image_path, conf_threshold=0.25)
    
    # Print results
    print(f"\nFound {len(detections)} celebrities:")
    for i, det in enumerate(detections, 1):
        print(f"{i}. Celebrity ID {det['celebrity_id']}")
        print(f"   Confidence: {det['confidence']:.2%}")
        print(f"   Location: {det['bbox']}")
    
    # Visualize
    visualize_detections(image_path, detections)
    
    return detections

# ============================================================================
# TEST ON VALIDATION IMAGES
# ============================================================================
print("\n" + "="*70)
print("Testing on validation images...")
print("="*70)

# Test on 3 validation images
test_images = [
    os.path.join(val_img_dir, f"val_{i:05d}.jpg") 
    for i in [0, 50, 100]
]

for i, img_path in enumerate(test_images, 1):
    if not os.path.exists(img_path):
        continue
    
    print(f"\n--- Test {i} ---")
    detections = process_image(img_path)

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("COMPLETE! Your Celebrity Detection System is Ready")
print("="*70)
print("\nCapabilities:")
print("✓ Detects multiple celebrities in one image")
print("✓ Identifies each celebrity by ID")
print("✓ Locates each celebrity with bounding boxes")
print("✓ Provides confidence scores")

print("\nUsage:")
print("  # Detect celebrities in any image")
print("  detections = process_image('path/to/your/image.jpg')")
print("\n  # Or just get detections without visualization")
print("  detections = detect_celebrities('path/to/your/image.jpg')")

print("\nModel location:")
print(f"  runs/detect/celeb_detector/weights/best.pt")
print("="*70)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/tegegne.r/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading existing dataset...
Dataset: /courses/IE7615.202610/students/tegegne.r/yolo_celeba_dataset
Classes: 47 celebrities
Training images: 1000
Validation images: 200

Training YOLOv8 for Celebrity Detection

Starting training on CPU...
Note: This will take 1-2 hours on CPU
Ultralytics 8.3.205 🚀 Python-3.10.16 torch-2.5.1 CPU (Intel Xeon CPU E5-2680 v4 @ 2.40GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_celeba_dataset/data.yaml, degrees

<Figure size 1200x800 with 1 Axes>


--- Test 2 ---

Processing: ./yolo_celeba_dataset/images/val/val_00050.jpg

image 1/1 /courses/IE7615.202610/students/tegegne.r/yolo_celeba_dataset/images/val/val_00050.jpg: 416x416 (no detections), 25.5ms
Speed: 1.4ms preprocess, 25.5ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 416)

Found 0 celebrities:


<Figure size 1200x800 with 1 Axes>


--- Test 3 ---

Processing: ./yolo_celeba_dataset/images/val/val_00100.jpg

image 1/1 /courses/IE7615.202610/students/tegegne.r/yolo_celeba_dataset/images/val/val_00100.jpg: 416x416 (no detections), 25.3ms
Speed: 1.3ms preprocess, 25.3ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 416)

Found 0 celebrities:


<Figure size 1200x800 with 1 Axes>


COMPLETE! Your Celebrity Detection System is Ready

Capabilities:
✓ Detects multiple celebrities in one image
✓ Identifies each celebrity by ID
✓ Locates each celebrity with bounding boxes
✓ Provides confidence scores

Usage:
  # Detect celebrities in any image
  detections = process_image('path/to/your/image.jpg')

  # Or just get detections without visualization
  detections = detect_celebrities('path/to/your/image.jpg')

Model location:
  runs/detect/celeb_detector/weights/best.pt


In [10]:
# Test on one of your validation images
test_img = "./yolo_celeba_dataset/images/val/val_00021.jpg"
detections = process_image(test_img)


Processing: ./yolo_celeba_dataset/images/val/val_00021.jpg

image 1/1 /courses/IE7615.202610/students/tegegne.r/yolo_celeba_dataset/images/val/val_00021.jpg: 416x416 (no detections), 25.4ms
Speed: 1.4ms preprocess, 25.4ms inference, 0.5ms postprocess per image at shape (1, 3, 416, 416)

Found 0 celebrities:


<Figure size 1200x800 with 1 Axes>

In [11]:
# ============================================================================
# CHECK AVAILABLE RESOURCES
# ============================================================================
import os
import psutil
import torch

print("="*70)
print("System Resources")
print("="*70)

# CPU info
cpu_count = os.cpu_count()
cpu_physical = psutil.cpu_count(logical=False)
print(f"\nCPU Cores:")
print(f"  Logical cores: {cpu_count}")
print(f"  Physical cores: {cpu_physical}")

# Memory
mem = psutil.virtual_memory()
print(f"\nMemory:")
print(f"  Total: {mem.total / (1024**3):.1f} GB")
print(f"  Available: {mem.available / (1024**3):.1f} GB")

# GPU check
print(f"\nGPU:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU count: {torch.cuda.device_count()}")
    print(f"  GPU name: {torch.cuda.get_device_name(0)}")
else:
    print(f"  MPS available (Mac): {torch.backends.mps.is_available()}")

print("="*70)

System Resources

CPU Cores:
  Logical cores: 28
  Physical cores: 28

Memory:
  Total: 503.3 GB
  Available: 496.5 GB

GPU:
  CUDA available: False
  MPS available (Mac): False


In [ ]:
# ============================================================================
# USE EXISTING ORIGINAL + AUGMENTED IMAGES FOR YOLOV8
# ============================================================================
import os, numpy as np, pandas as pd
from PIL import Image
import random

YOLO_DATASET_DIR = "./yolo_celeba_dataset_enhanced"
os.makedirs(f"{YOLO_DATASET_DIR}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/labels/val", exist_ok=True)

print("="*70)
print("Using ALL existing images: Original + Augmented + Composites")
print("="*70)

# Use df_target which already includes both original and augmented images
print(f"\nTotal images available: {len(df_target)}")
print(f"  - Original CelebA images")
print(f"  - Augmented images you created earlier")

# ============================================================================
# FUNCTION: SINGLE CELEBRITY IMAGE WITH BBOX
# ============================================================================
def create_single_celeb_sample(celeb_id, img_filepath):
    """
    Load existing image (original or augmented) and create YOLO label
    """
    # Handle both full paths (augmented) and relative paths (original)
    if os.path.isabs(img_filepath):
        full_path = img_filepath
    else:
        full_path = os.path.join(IMG_DIR, img_filepath)
    
    try:
        img = Image.open(full_path).convert('RGB')
        img = img.resize((640, 640))  # Standard YOLO size
        
        # Full image bbox (celebrity takes up whole image)
        class_id = class_map[celeb_id]
        box = [class_id, 0.5, 0.5, 1.0, 1.0]
        
        return img, [box]
    except Exception as e:
        print(f"Error loading {img_filepath}: {e}")
        return None, None

# ============================================================================
# COMPOSITE FUNCTION (from before)
# ============================================================================
def create_composite_image(df_source, canvas_size=(640, 640), min_celebs=2, max_celebs=5):
    """Your existing composite function"""
    canvas = Image.new('RGB', canvas_size, color=(128, 128, 128))
    
    num_celebs = random.randint(min_celebs, max_celebs)
    sampled_celebs = random.sample(target_celebs, min(num_celebs, len(target_celebs)))
    
    boxes = []
    occupied_regions = []
    
    for celeb_id in sampled_celebs:
        celeb_imgs = df_source[df_source['identity'] == celeb_id]['filename'].tolist()
        if not celeb_imgs:
            continue
        
        img_path = random.choice(celeb_imgs)
        
        # Handle both path types
        if os.path.isabs(img_path):
            full_path = img_path
        else:
            full_path = os.path.join(IMG_DIR, img_path)
        
        try:
            img = Image.open(full_path).convert('RGB')
        except:
            continue
        
        scale = random.uniform(0.3, 0.7)
        new_w = int(img.width * scale)
        new_h = int(img.height * scale)
        img_resized = img.resize((new_w, new_h))
        
        placed = False
        for attempt in range(10):
            x = random.randint(0, max(0, canvas_size[0] - new_w))
            y = random.randint(0, max(0, canvas_size[1] - new_h))
            
            overlap = False
            for (ox, oy, ow, oh) in occupied_regions:
                if not (x + new_w < ox or x > ox + ow or 
                       y + new_h < oy or y > oy + oh):
                    overlap = True
                    break
            
            if not overlap or attempt == 9:
                placed = True
                break
        
        if placed:
            canvas.paste(img_resized, (x, y))
            occupied_regions.append((x, y, new_w, new_h))
            
            x_center = (x + new_w / 2) / canvas_size[0]
            y_center = (y + new_h / 2) / canvas_size[1]
            width = new_w / canvas_size[0]
            height = new_h / canvas_size[1]
            
            class_id = class_map[celeb_id]
            boxes.append([class_id, x_center, y_center, width, height])
    
    return canvas, boxes

# ============================================================================
# SPLIT DATA
# ============================================================================
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_target,
    test_size=0.15,
    random_state=42,
    stratify=df_target['identity']
)

print(f"\nData split:")
print(f"  Train: {len(train_df)} images")
print(f"  Val:   {len(val_df)} images")

# ============================================================================
# GENERATE TRAINING SET
# ============================================================================
train_count = 0

print("\n" + "="*70)
print("Generating TRAINING set...")
print("="*70)

# 1. All single-celebrity images (original + augmented)
print(f"\nAdding {len(train_df)} single-celebrity images...")
for idx, row in train_df.iterrows():
    img, boxes = create_single_celeb_sample(row['identity'], row['filename'])
    
    if img is None:
        continue
    
    img_name = f"train_single_{train_count:05d}.jpg"
    img.save(os.path.join(YOLO_DATASET_DIR, "images", "train", img_name))
    
    label_name = f"train_single_{train_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "train", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    train_count += 1
    
    if train_count % 200 == 0:
        print(f"  Processed {train_count}/{len(train_df)} images...")

# 2. Composite multi-celebrity images
print(f"\nAdding 2000 composite images...")
NUM_COMPOSITES = 2000

for i in range(NUM_COMPOSITES):
    composite, boxes = create_composite_image(train_df)
    
    if len(boxes) == 0:
        continue
    
    img_name = f"train_composite_{train_count:05d}.jpg"
    composite.save(os.path.join(YOLO_DATASET_DIR, "images", "train", img_name))
    
    label_name = f"train_composite_{train_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "train", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    train_count += 1
    
    if (i + 1) % 500 == 0:
        print(f"  Generated {i+1}/{NUM_COMPOSITES} composites...")

print(f"\nTotal training images: {train_count}")

# ============================================================================
# GENERATE VALIDATION SET
# ============================================================================
val_count = 0

print("\n" + "="*70)
print("Generating VALIDATION set...")
print("="*70)

# Single-celebrity validation
print(f"\nAdding {len(val_df)} single-celebrity validation images...")
for idx, row in val_df.iterrows():
    img, boxes = create_single_celeb_sample(row['identity'], row['filename'])
    
    if img is None:
        continue
    
    img_name = f"val_single_{val_count:05d}.jpg"
    img.save(os.path.join(YOLO_DATASET_DIR, "images", "val", img_name))
    
    label_name = f"val_single_{val_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "val", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    val_count += 1

# Composite validation
print(f"\nAdding 400 composite validation images...")
for i in range(400):
    composite, boxes = create_composite_image(val_df)
    
    if len(boxes) == 0:
        continue
    
    img_name = f"val_composite_{val_count:05d}.jpg"
    composite.save(os.path.join(YOLO_DATASET_DIR, "images", "val", img_name))
    
    label_name = f"val_composite_{val_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "val", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    val_count += 1

print(f"\nTotal validation images: {val_count}")

# ============================================================================
# CREATE data.yaml
# ============================================================================
yaml_content = f"""path: {os.path.abspath(YOLO_DATASET_DIR)}
train: images/train
val: images/val

nc: {NUM_CELEBS}
names: {list(range(NUM_CELEBS))}
"""

with open(os.path.join(YOLO_DATASET_DIR, "data.yaml"), 'w') as f:
    f.write(yaml_content)

pd.DataFrame({
    'class_id': list(reverse_class_map.keys()),
    'celebrity_id': list(reverse_class_map.values())
}).to_csv(os.path.join(YOLO_DATASET_DIR, "class_mapping.csv"), index=False)

print("\n" + "="*70)
print("ENHANCED DATASET COMPLETE!")
print("="*70)
print(f"Dataset: {os.path.abspath(YOLO_DATASET_DIR)}")
print(f"Training images: {train_count}")
print(f"  - Single-celebrity (original + augmented): ~{len(train_df)}")
print(f"  - Multi-celebrity composites: ~{NUM_COMPOSITES}")
print(f"Validation images: {val_count}")
print(f"\nExpected mAP50: 35-55% (vs 18% before)")
print("="*70)

In [12]:
# ============================================================================
# ENHANCED DATASET: ALL IMAGES (ORIGINAL + AUGMENTED + COMPOSITES)
# ============================================================================
import os, numpy as np, pandas as pd
from PIL import Image
import random

YOLO_DATASET_DIR = "./yolo_celeba_dataset_enhanced"
os.makedirs(f"{YOLO_DATASET_DIR}/images/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/images/val", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{YOLO_DATASET_DIR}/labels/val", exist_ok=True)

print("="*70)
print("Creating Enhanced Dataset with ALL Available Images")
print("="*70)
print(f"Total images in df_target: {len(df_target)}")
print(f"  - Original CelebA images")
print(f"  - Augmented celebrity images")
print(f"Celebrities: {len(target_celebs)}")

# ============================================================================
# FUNCTION: SINGLE CELEBRITY IMAGE
# ============================================================================
def create_single_celeb_sample(celeb_id, img_filepath):
    """Load existing image and create YOLO label with full-image bbox"""
    # Handle both full paths (augmented) and relative paths (original)
    if os.path.isabs(img_filepath):
        full_path = img_filepath
    else:
        full_path = os.path.join(IMG_DIR, img_filepath)
    
    try:
        img = Image.open(full_path).convert('RGB')
        img = img.resize((640, 640))
        
        # Full image bbox
        class_id = class_map[celeb_id]
        box = [class_id, 0.5, 0.5, 1.0, 1.0]  # centered, full size
        
        return img, [box]
    except Exception as e:
        return None, None

# ============================================================================
# FUNCTION: COMPOSITE MULTI-CELEBRITY IMAGE
# ============================================================================
def create_composite_image(df_source, canvas_size=(640, 640), min_celebs=2, max_celebs=5):
    """Create composite with multiple celebrities"""
    canvas = Image.new('RGB', canvas_size, color=(128, 128, 128))
    
    num_celebs = random.randint(min_celebs, max_celebs)
    sampled_celebs = random.sample(target_celebs, min(num_celebs, len(target_celebs)))
    
    boxes = []
    occupied_regions = []
    
    for celeb_id in sampled_celebs:
        celeb_imgs = df_source[df_source['identity'] == celeb_id]['filename'].tolist()
        if not celeb_imgs:
            continue
        
        img_path = random.choice(celeb_imgs)
        
        if os.path.isabs(img_path):
            full_path = img_path
        else:
            full_path = os.path.join(IMG_DIR, img_path)
        
        try:
            img = Image.open(full_path).convert('RGB')
        except:
            continue
        
        # Random scale
        scale = random.uniform(0.25, 0.75)
        new_w = int(img.width * scale)
        new_h = int(img.height * scale)
        img_resized = img.resize((new_w, new_h))
        
        # Find position
        placed = False
        for attempt in range(10):
            x = random.randint(0, max(0, canvas_size[0] - new_w))
            y = random.randint(0, max(0, canvas_size[1] - new_h))
            
            # Check overlap
            overlap = False
            for (ox, oy, ow, oh) in occupied_regions:
                if not (x + new_w < ox or x > ox + ow or 
                       y + new_h < oy or y > oy + oh):
                    overlap = True
                    break
            
            if not overlap or attempt == 9:
                placed = True
                break
        
        if placed:
            canvas.paste(img_resized, (x, y))
            occupied_regions.append((x, y, new_w, new_h))
            
            # YOLO format
            x_center = (x + new_w / 2) / canvas_size[0]
            y_center = (y + new_h / 2) / canvas_size[1]
            width = new_w / canvas_size[0]
            height = new_h / canvas_size[1]
            
            class_id = class_map[celeb_id]
            boxes.append([class_id, x_center, y_center, width, height])
    
    return canvas, boxes

# ============================================================================
# SPLIT DATA
# ============================================================================
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_target,
    test_size=0.15,
    random_state=42,
    stratify=df_target['identity']
)

print(f"\nData split:")
print(f"  Train: {len(train_df)} single images")
print(f"  Val:   {len(val_df)} single images")

# ============================================================================
# GENERATE TRAINING SET
# ============================================================================
train_count = 0

print("\n" + "="*70)
print("GENERATING TRAINING SET")
print("="*70)

# Part 1: All single-celebrity images
print(f"\n[1/2] Adding all single-celebrity images (original + augmented)...")
for idx, row in train_df.iterrows():
    img, boxes = create_single_celeb_sample(row['identity'], row['filename'])
    
    if img is None:
        continue
    
    img_name = f"train_single_{train_count:05d}.jpg"
    img.save(os.path.join(YOLO_DATASET_DIR, "images", "train", img_name))
    
    label_name = f"train_single_{train_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "train", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    train_count += 1
    
    if train_count % 200 == 0:
        print(f"  Processed {train_count}/{len(train_df)}...")

print(f"✓ Added {train_count} single-celebrity images")

# Part 2: Generate composite images
print(f"\n[2/2] Generating composite multi-celebrity images...")
NUM_COMPOSITES = 3000  # Generate 3000 composites

composite_start = train_count

for i in range(NUM_COMPOSITES):
    composite, boxes = create_composite_image(train_df)
    
    if len(boxes) == 0:
        continue
    
    img_name = f"train_composite_{train_count:05d}.jpg"
    composite.save(os.path.join(YOLO_DATASET_DIR, "images", "train", img_name))
    
    label_name = f"train_composite_{train_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "train", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    train_count += 1
    
    if (i + 1) % 500 == 0:
        print(f"  Generated {i+1}/{NUM_COMPOSITES}...")

print(f"✓ Added {train_count - composite_start} composite images")
print(f"\n✓ Total training images: {train_count}")

# ============================================================================
# GENERATE VALIDATION SET
# ============================================================================
val_count = 0

print("\n" + "="*70)
print("GENERATING VALIDATION SET")
print("="*70)

# Part 1: Single-celebrity validation
print(f"\n[1/2] Adding single-celebrity validation images...")
for idx, row in val_df.iterrows():
    img, boxes = create_single_celeb_sample(row['identity'], row['filename'])
    
    if img is None:
        continue
    
    img_name = f"val_single_{val_count:05d}.jpg"
    img.save(os.path.join(YOLO_DATASET_DIR, "images", "val", img_name))
    
    label_name = f"val_single_{val_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "val", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    val_count += 1

print(f"✓ Added {val_count} single-celebrity images")

# Part 2: Composite validation
print(f"\n[2/2] Generating composite validation images...")
NUM_VAL_COMPOSITES = 500

val_composite_start = val_count

for i in range(NUM_VAL_COMPOSITES):
    composite, boxes = create_composite_image(val_df)
    
    if len(boxes) == 0:
        continue
    
    img_name = f"val_composite_{val_count:05d}.jpg"
    composite.save(os.path.join(YOLO_DATASET_DIR, "images", "val", img_name))
    
    label_name = f"val_composite_{val_count:05d}.txt"
    with open(os.path.join(YOLO_DATASET_DIR, "labels", "val", label_name), 'w') as f:
        for box in boxes:
            f.write(f"{box[0]} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
    
    val_count += 1
    
    if (i + 1) % 100 == 0:
        print(f"  Generated {i+1}/{NUM_VAL_COMPOSITES}...")

print(f"✓ Added {val_count - val_composite_start} composite images")
print(f"\n✓ Total validation images: {val_count}")

# ============================================================================
# CREATE data.yaml
# ============================================================================
yaml_content = f"""path: {os.path.abspath(YOLO_DATASET_DIR)}
train: images/train
val: images/val

nc: {NUM_CELEBS}
names: {list(range(NUM_CELEBS))}
"""

with open(os.path.join(YOLO_DATASET_DIR, "data.yaml"), 'w') as f:
    f.write(yaml_content)

# Save mapping
pd.DataFrame({
    'class_id': list(reverse_class_map.keys()),
    'celebrity_id': list(reverse_class_map.values())
}).to_csv(os.path.join(YOLO_DATASET_DIR, "class_mapping.csv"), index=False)

print("\n" + "="*70)
print("ENHANCED DATASET COMPLETE!")
print("="*70)
print(f"Location: {os.path.abspath(YOLO_DATASET_DIR)}")
print(f"\nDataset Summary:")
print(f"  Training images: {train_count}")
print(f"    - Single-celebrity: ~{composite_start}")
print(f"    - Multi-celebrity:  ~{train_count - composite_start}")
print(f"  Validation images: {val_count}")
print(f"    - Single-celebrity: ~{val_composite_start}")
print(f"    - Multi-celebrity:  ~{val_count - val_composite_start}")
print(f"  Total: {train_count + val_count} images")
print(f"  Classes: {NUM_CELEBS} celebrities")
print("\n" + "="*70)
print("Ready for optimized training with 28 CPUs!")
print("="*70)

Creating Enhanced Dataset with ALL Available Images
Total images in df_target: 1328
  - Original CelebA images
  - Augmented celebrity images
Celebrities: 47

Data split:
  Train: 1128 single images
  Val:   200 single images

GENERATING TRAINING SET

[1/2] Adding all single-celebrity images (original + augmented)...
  Processed 200/1128...
  Processed 400/1128...
  Processed 600/1128...
  Processed 800/1128...
  Processed 1000/1128...
✓ Added 1128 single-celebrity images

[2/2] Generating composite multi-celebrity images...
  Generated 500/3000...
  Generated 1000/3000...
  Generated 1500/3000...
  Generated 2000/3000...
  Generated 2500/3000...
  Generated 3000/3000...
✓ Added 3000 composite images

✓ Total training images: 4128

GENERATING VALIDATION SET

[1/2] Adding single-celebrity validation images...
✓ Added 200 single-celebrity images

[2/2] Generating composite validation images...
  Generated 100/500...
  Generated 200/500...
  Generated 300/500...
  Generated 400/500...
  G

In [16]:
# Run this on your VM to create a zip file
import shutil

# Zip the enhanced dataset
shutil.make_archive('yolo_celeba_dataset_enhanced', 'zip', YOLO_DATASET_DIR)
print("Dataset zipped! Download: yolo_celeba_dataset_enhanced.zip")

# Also save important files
import pandas as pd
pd.DataFrame({
    'class_id': list(reverse_class_map.keys()),
    'celebrity_id': list(reverse_class_map.values())
}).to_csv('class_mapping.csv', index=False)

Dataset zipped! Download: yolo_celeba_dataset_enhanced.zip
